# ***Model 5: Batch Optimization CNN Model using CIFAR-10 DataSet***

Apply CNN with Batch Normalization, Dropout, and Adam optimizer and evaluate if Batch Normalization will make the training faster or more stable.

In [ ]:
import os
from pathlib import Path

# If notebook is opened from notebooks/ folder, move to project root
if Path.cwd().name == "notebooks":
    os.chdir("..")

print("Current directory:", Path.cwd())
print("Project files:", os.listdir("."))

In [ ]:
MODEL_ID = 5
MODEL_NAME = "batchnorm_cnn"

In [ ]:
# Imports tensorflow.Keras lib
import importlib

from keras import layers
from keras.backend import clear_session
from keras.optimizers import SGD

# Imports seaborn lib
import seaborn as sns

# Imports sklearn lib
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score
from sklearn.metrics import confusion_matrix

# Imports custom files aka config, experiment files, etc
from src.training.trainer import compile_model, train_model, save_model
from src.training.evaluator import (
    evaluate_model,
    get_predictions,
    calculate_precision_recall,
    save_confusion_matrix
)
from src import config
from src.experiment_tracker import save_experiment_results  
from training.model_builder import build_dropout_cnn, build_batchnorm_cnn

In [ ]:
# Load the data fromm cache using the load_data function from data_loader.py
from src.data_loader import load_data

x_train, y_train, x_val, y_val, x_test, y_test = load_data()

In [ ]:
# Clear any previous TensorFlow models from memory as TensorFlow can accumulate unnecessary objects in memory,
# especially if we are creating many models while tuning hyperparameters.
clear_session()

In [ ]:
# compile the model using the compile_model function from trainer.py
batchnorm_model = build_batchnorm_cnn()
batchnorm_model.summary()

In [ ]:
# appluy the Adam optimizer to the model using the compile_model function from trainer.py
adam_optimizer = keras.optimizers.Adam(
    learning_rate=config.LEARNING_RATE,
)

batchnorm_model = compile_model(
    model=batchnorm_model,
    optimizer=adam_optimizer
)

In [ ]:
# Train the model using the train_model function from trainer.py
history, train_time = train_model(
    model=batchnorm_model,
    x_train=x_train,
    y_train=y_train,
    x_val=x_val,
    y_val=y_val,
    batch_size=config.BATCH_SIZE,
    epochs=config.EPOCHS
)

In [ ]:
# Sanity check: confirm we're using the new 70/15/15 split (15% of 60,000 = 9,000) and not the original Keras default test set (10,000)
print("x_test shape:", x_test.shape)
print("Expected: ~9000 (15% of 60000)")

# Evaluate the baseline model on the held-out test set 
test_loss, test_accuracy = evaluate_model(
    model=batchnorm_model,
    x_test=x_test,
    y_test=y_test
)

test_labels, test_predictions, y_pred_probs = get_predictions(
    model=batchnorm_model,
    x_test=x_test,
    y_test=y_test
)

precision, recall = calculate_precision_recall(
    test_labels=test_labels,
    test_predictions=test_predictions
)

print("Test accuracy:", test_accuracy)
print("Precision:", precision)
print("Recall:", recall)

In [ ]:
model_path = save_model(
    model=batchnorm_model,
    model_name=f"model_{MODEL_ID}_{MODEL_NAME}",
    models_dir=config.MODELS_DIR
)

In [ ]:
# Save the confusion matrix to a file
confusion_matrix_path = (
    f"{config.CONFUSION_MATRIX_DIR}/model_{MODEL_ID}_{MODEL_NAME}_confusion_matrix.png"
)

save_confusion_matrix(
    y_true=test_labels,
    y_pred=test_predictions,
    class_names=config.CLASS_NAMES,
    output_path=confusion_matrix_path,
    title=f"Model {MODEL_ID} - Batch Normalization CNN Confusion Matrix using CIFAR-10 dataset"
)

In [ ]:
experiment = {
    "model_id": MODEL_ID,
    "model_name": MODEL_NAME,
    "architecture": "Conv-Conv-Pool x2 + BatchNorm + Dense(64) + Dropout(0.5)",
    "optimizer": "Adam",
    "learning_rate": config.LEARNING_RATE,
    "batch_size": config.BATCH_SIZE,
    "epochs": len(history.history["accuracy"]),
    "train_accuracy": history.history["accuracy"][-1],
    "validation_accuracy": history.history["val_accuracy"][-1],
    "test_accuracy": test_accuracy,
    "precision": precision,
    "recall": recall,
    "train_loss": history.history["loss"][-1],
    "validation_loss": history.history["val_loss"][-1],
    "test_loss": test_loss,
    "train_time_seconds": train_time,
    "model_path": model_path,
    "confusion_matrix_path": confusion_matrix_path,
    "notes": "Model 5 tests whether Batch Normalization improves stability and performance"}

results_df = save_experiment_results(
    experiment,
    config.RESULTS_PATH
)

results_df